# Users Feature engineering

We will create the columns that are intersing for our users

In [ ]:
import pandas as pd

In [ ]:
sessions = pd.read_csv('../data/raw/sessions_raw.csv')
users = pd.read_csv('../data/raw/users_raw.csv')
flights = pd.read_csv('../data/raw/flights_raw.csv')
hotels = pd.read_csv('../data/processed/hotels_pocessed.csv')

id from trips that were cancelled

In [ ]:
cancelled_trip_ids = sessions[sessions['cancellation']]['trip_id']

## Columns directly from users table

In [ ]:
users.columns

directly took columns from user table

In [ ]:
user_features = users[['user_id', 'gender', 'married', 'has_children', 'home_country']].copy()

## Tenure

In [ ]:
user_features['tenure'] = (pd.Timestamp.today() - pd.to_datetime(users['sign_up_date'])).dt.days # feature column

Idea: From this column create group of new customers.

## Age


In [ ]:
users.columns

In [ ]:
user_features['age']  = (pd.Timestamp.today() - pd.to_datetime(users['birthdate'])).dt.days//365

In [ ]:
user_features

## Overcarrier

In [ ]:
flights

In [ ]:
flights_w_users = pd.merge(flights, sessions[['trip_id','user_id']], on='trip_id', how='left')

filter out valid trips and calculate average bags per seat

In [ ]:
seats_avg = flights_w_users[~flights_w_users['trip_id'].isin(cancelled_trip_ids)].groupby('user_id')['seats'].mean()
seats_avg.name = "seats_avg"  # feature column


In [ ]:
checked_bags_avg = flights_w_users[~flights_w_users['trip_id'].isin(cancelled_trip_ids)].groupby('user_id')['checked_bags'].mean()
checked_bags_avg.name = "checked_bags_avg"  # feature column

In [ ]:
bags_per_seat = checked_bags_avg/seats_avg

In [ ]:
overcarrier = bags_per_seat > 1  # feature column

overcarrier.name = "overcarrier"

## Frequent travelers

In [ ]:
total_trips  = sessions.groupby('user_id')['trip_id'].nunique() # feature column
total_trips.name = 'total_trips'

In [ ]:
number_of_cancellations = sessions.groupby('user_id')['cancellation'].sum()

In [ ]:
cancellation_rate  = number_of_cancellations/total_trips  # feature column
cancellation_rate.name = 'cancellation_rate'

## Frequent destination

In [ ]:
unique_flight_destination = flights_w_users.groupby('user_id')['destination'].nunique()
unique_flight_destination.name = 'unique_flight_destination' # feature column

In [ ]:
taken_flights = total_trips - number_of_cancellations
taken_flights.name = 'taken_flights' # feature column

In [ ]:
frequent_destinations = unique_flight_destination/taken_flights  # feature column
frequent_destinations.name = 'frequent_destinations'

## Flight/Hotel booked

In [ ]:
# Get unique user IDs from flights_w_users (excluding cancelled trips)
users_with_booked_flights = flights_w_users[~flights_w_users['trip_id'].isin(cancelled_trip_ids)]['user_id'].unique()
booked_flight = pd.DataFrame({"user_id": user_features['user_id'], "booked_flight": user_features['user_id'].isin(users_with_booked_flights)}) # feature column


In [ ]:
hotels_w_users = pd.merge(hotels, sessions[['trip_id','user_id']], on='trip_id', how='left')

users_with_booked_hotels = hotels_w_users[~hotels_w_users['trip_id'].isin(cancelled_trip_ids)]['user_id'].unique()

booked_hotel = pd.DataFrame({"user_id": user_features['user_id'], "booked_hotel": user_features['user_id'].isin(users_with_booked_hotels)})  # feature column

## Booking rate

In [ ]:
total_sessions = sessions[~sessions['trip_id'].isin(cancelled_trip_ids)].groupby("user_id")['session_id'].count()
total_sessions.name = 'total_sessions'

In [ ]:
nan_sessions = sessions[~sessions['trip_id'].isin(cancelled_trip_ids)].groupby('user_id')['trip_id'].apply(lambda x: x.isna().sum())

In [ ]:
sessions_booking_rate = 1 - nan_sessions/total_sessions # feature column
sessions_booking_rate.name = 'sessions_booking_rate'

## Percentage discounted trip

In [ ]:
sessions

In [ ]:
valid_bookings = sessions[(~sessions['trip_id'].isin(cancelled_trip_ids)) & (~sessions['trip_id'].isna())].copy()

In [ ]:
valid_bookings['discount']  = valid_bookings["flight_discount"] | valid_bookings["hotel_discount"]

In [ ]:
discount_booking_rate = valid_bookings.groupby("user_id")['discount'].mean() # feature column
discount_booking_rate.name = 'discount_booking_rate'

# Create user features table by joining all the features together

In [ ]:
user_features

Example of merge

In [ ]:
user_features.merge(discount_booking_rate, on='user_id', how='left')

If there is a "No name" error you can solve it with the following line

In [ ]:
sessions_booking_rate.name = 'sessions_booking_rate'

In [ ]:
new_users = user_features.merge(sessions_booking_rate, on='user_id', how='left')

# Merge all Feature columns together

In [ ]:
seats_avg

In [ ]:
user_features

In [ ]:
user_features_new = (user_features.merge(seats_avg, on='user_id', how='left')
 .merge(checked_bags_avg, on='user_id', how='left')
 .merge(overcarrier, on='user_id', how='left')
 .merge(total_trips, on='user_id', how='left')
 .merge(cancellation_rate, on='user_id', how='left')
 .merge(unique_flight_destination, on='user_id', how='left')
 .merge(taken_flights, on='user_id', how='left')
 .merge(frequent_destinations, on='user_id', how='left')
 .merge(sessions_booking_rate, on='user_id', how='left')
 .merge(discount_booking_rate, on='user_id', how='left')
 .merge(booked_flight, on='user_id', how='left')
 .merge(booked_hotel, on='user_id', how='left'))

In [ ]:
user_features_new

In [ ]:
user_features_new.to_csv("data/processed/user_features.csv", index = False)

In [ ]:
df = pd.read_csv("../data/processed/user_features.csv")
df